# **`Vendor Performance Analysis`**

In [1]:
##  Import Libraries & Setup

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

Mounted at /content/drive


In [2]:
##  Define Data Path

base_path = '/content/drive/MyDrive/Vendor-Performance-Analysis/data/raw/'

In [5]:

pd.read_csv(base_path + 'begin_inventory.csv').columns

##    Data Loading
##    Load all required datasets from raw folder.

# SALES
sales = pd.read_csv(
    base_path + 'sales.csv',
    usecols=['InventoryId','Brand','SalesQuantity','SalesDollars','VendorNo']
)

# PURCHASES
purchases = pd.read_csv(
    base_path + 'purchases.csv',
    usecols=['InventoryId','VendorNumber','PurchasePrice']
)

# PRICES
prices = pd.read_csv(
    base_path + 'purchase_prices.csv',
    usecols=['Brand','PurchasePrice']
)

# BEGIN INVENTORY ( FIXED: onHand)
begin_inv = pd.read_csv(
    base_path + 'begin_inventory.csv',
    usecols=['InventoryId','onHand']
)

# END INVENTORY ( FIXED: onHand)
end_inv = pd.read_csv(
    base_path + 'end_inventory.csv',
    usecols=['InventoryId','onHand']
)

# VENDOR
vendor = pd.read_csv(
    base_path + 'vendor_invoice.csv',
    usecols=['VendorNumber','Freight']
)

print("Data loaded")

✅ Data loaded


In [6]:
##  Data Cleaning
##  Standardize column names and fix inconsistencies.

sales.columns = sales.columns.str.strip().str.lower()
purchases.columns = purchases.columns.str.strip().str.lower()
prices.columns = prices.columns.str.strip().str.lower()
begin_inv.columns = begin_inv.columns.str.strip().str.lower()
end_inv.columns = end_inv.columns.str.strip().str.lower()
vendor.columns = vendor.columns.str.strip().str.lower()

sales.rename(columns={'vendorno': 'vendornumber'}, inplace=True)

print("Columns cleaned")

✅ Columns cleaned


In [7]:
##  Remove Duplicates
##  Prepare clean datasets for merging.

purchases = purchases.drop_duplicates(['inventoryid','vendornumber'])
prices = prices.drop_duplicates(['brand'])
begin_inv = begin_inv.drop_duplicates(['inventoryid'])
end_inv = end_inv.drop_duplicates(['inventoryid'])
vendor = vendor.drop_duplicates(['vendornumber'])

print("Duplicates removed")

✅ Duplicates removed


In [8]:
## Data Merging
## Combine multiple datasets into a single dataset.

df = sales.merge(purchases, on=['inventoryid','vendornumber'], how='left')

df = df.merge(prices, on='brand', how='left')

df = df.merge(begin_inv, on='inventoryid', how='left').rename(columns={'onhand':'begin_inventory'})

df = df.merge(end_inv, on='inventoryid', how='left').rename(columns={'onhand':'end_inventory'})

df = df.merge(vendor, on='vendornumber', how='left')

print("Merge completed")

Merge completed


In [9]:
##  KPI Creation
##  Calculate key business metrics.

df['revenue'] = df['salesdollars']
df['cost'] = df['purchaseprice_x'] * df['salesquantity']
df['profit'] = df['revenue'] - df['cost']
df['profit_margin'] = df['profit'] / df['revenue'].replace(0, 1)

df['inventory_turnover'] = df['salesquantity'] / (
    ((df['begin_inventory'] + df['end_inventory']) / 2).replace(0, 1)
)

print("KPIs created")

KPIs created


In [10]:
## Data Validation
## Check dataset quality and structure.

print(df.shape)
df.isnull().sum()
df.fillna(0, inplace=True)
df.dtypes
df.head()

(12825363, 15)


,inventoryid,brand,salesquantity,salesdollars,vendornumber,purchaseprice_x,purchaseprice_y,begin_inventory,end_inventory,freight,revenue,cost,profit,profit_margin,inventory_turnover
0,1_HARDERSFIELD_1004,1004,1,16.49,12546,10.65,10.65,17.0,0.0,3506.08,16.49,10.65,5.84,0.354154,0.0
1,1_HARDERSFIELD_1004,1004,2,32.98,12546,10.65,10.65,17.0,0.0,3506.08,32.98,21.30,11.68,0.354154,0.0
2,1_HARDERSFIELD_1004,1004,1,16.49,12546,10.65,10.65,17.0,0.0,3506.08,16.49,10.65,5.84,0.354154,0.0
3,1_HARDERSFIELD_1004,1004,1,14.49,12546,10.65,10.65,17.0,0.0,3506.08,14.49,10.65,3.84,0.265010,0.0
4,1_HARDERSFIELD_1005,1005,2,69.98,12546,27.34,27.34,7.0,0.0,3506.08,69.98,54.68,15.30,0.218634,0.0


In [ ]:
df.isnull().sum()

,0
inventoryid,0
brand,0
salesquantity,0
salesdollars,0
vendornumber,0
purchaseprice_x,190978
purchaseprice_y,0
begin_inventory,1141295
end_inventory,724183
freight,3


In [11]:
df.fillna(0, inplace=True)

print("Missing values handled")

Missing values handled


In [ ]:
df.dtypes

,0
inventoryid,object
brand,int64
salesquantity,int64
salesdollars,float64
vendornumber,int64
purchaseprice_x,float64
purchaseprice_y,float64
begin_inventory,float64
end_inventory,float64
freight,float64


In [ ]:
df.head()

,inventoryid,brand,salesquantity,salesdollars,vendornumber,purchaseprice_x,purchaseprice_y,begin_inventory,end_inventory,freight,revenue,cost,profit,profit_margin,inventory_turnover
0,1_HARDERSFIELD_1004,1004,1,16.49,12546,10.65,10.65,17.0,0.0,3506.08,16.49,10.65,5.84,0.354154,0.0
1,1_HARDERSFIELD_1004,1004,2,32.98,12546,10.65,10.65,17.0,0.0,3506.08,32.98,21.30,11.68,0.354154,0.0
2,1_HARDERSFIELD_1004,1004,1,16.49,12546,10.65,10.65,17.0,0.0,3506.08,16.49,10.65,5.84,0.354154,0.0
3,1_HARDERSFIELD_1004,1004,1,14.49,12546,10.65,10.65,17.0,0.0,3506.08,14.49,10.65,3.84,0.265010,0.0
4,1_HARDERSFIELD_1005,1005,2,69.98,12546,27.34,27.34,7.0,0.0,3506.08,69.98,54.68,15.30,0.218634,0.0


In [13]:
## Save Processed Data
## Export final dataset for Power BI.
os.makedirs('/content/drive/MyDrive/Vendor-Performance-Analysis/data/processed', exist_ok=True)

output_path = '/content/drive/MyDrive/Vendor-Performance-Analysis/data/processed/final_dataset.csv'

df.to_csv(output_path, index=False)

print("FINAL DATASET SAVED ")

FINAL DATASET SAVED 
